In [1]:
print('a')

a


In [2]:
%pwd

'c:\\Softwares\\github\\End-to-End-Medical-Chat-Bot-Generative-AI\\research'

In [4]:
import os
os.chdir("../")
%pwd

'c:\\Softwares\\github\\End-to-End-Medical-Chat-Bot-Generative-AI'

In [5]:
from langchain.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
from langchain.document_loaders import PyPDFLoader
import os

def load_pdf_file(data):
    print(f'Loading data from directory: {data}')
    data_path = os.path.join(os.getcwd(), data)
    
    try:
        files = os.listdir(data_path)
        pdf_files = [f for f in files if f.lower().endswith('.pdf')]
        
        if not pdf_files:
            print("No PDF files found")
            return []
            
        documents = []
        for pdf_file in pdf_files:
            file_path = os.path.join(data_path, pdf_file)
            try:
                # Try loading individual PDF file
                loader = PyPDFLoader(file_path)
                file_documents = loader.load()
                print(f"Successfully loaded {pdf_file}: {len(file_documents)} pages")
                documents.extend(file_documents)
            except Exception as e:
                print(f"Error loading {pdf_file}: {str(e)}")
                
        return documents
            
    except Exception as e:
        print(f'Error accessing directory: {str(e)}')
        return []

# Test the loader
extracted_data = load_pdf_file(data='Data')


In [27]:
# Split the Data into chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

print(f'Final number of documents loaded: {len(text_split(extracted_data))}')

Final number of documents loaded: 5859


In [ ]:
text_chunks = text_split(extracted_data)
text_chunks

In [30]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_hugging_face_embeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'):
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

In [ ]:
embeddings = download_hugging_face_embeddings()

In [40]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
from dotenv import load_dotenv
load_dotenv()
import os   

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medicalbot-index"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ),
)


PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'efed4bff9080d6ba95086f91929555ca', 'date': 'Sat, 25 Oct 2025 10:32:21 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [41]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [37]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    text_chunks,
    embeddings,
    index_name=index_name,
)

In [ ]:
# Load Existing Index

from langchain_pinecone import PineconeVectorStore
# Embeddings 

docsearch = PineconeVectorStore.from_existing_index(
    embeddings,
    index_name=index_name,
)

In [39]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retrived_docs = retriever.get_relevant_documents("What is Acne?")
retrived_docs

[Document(id='a877046e-34ed-403b-9953-f7f7039d991a', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'c:\\Softwares\\github\\End-to-End-Medical-Chat-Bot-Generative-AI\\Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='d085185b-f2e6-419c-9540-0a3259247d84', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'c:\\Softwares\\github\\End-to-End-Medical-Chat-Bot-Generative-AI\\Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which th

In [42]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens = 500)


In [52]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an medical expert assistant for question-answering tasks"
    "Use the following pieces of retrieved context to answer the question at the end. "
    "If you don't know the answer, just say that you don't know, don't try to make up an answer."
    "Use three sentences maximum and be precise." 
    "\n\n"
    "{context}\n"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Answer the question: {input}")
])

In [54]:
question_answering_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=question_answering_chain)


In [56]:
response = rag_chain.invoke({"input":"What is Acne?"})
print(response["answer"])



Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris, the medical term for common acne, is the most common skin disease affecting nearly 17 million people in the United States.


In [ ]:
response = rag_chain.invoke({"input":"What is Stats?"})
print(response["answer"])